# Volve 15/9-F-4 — Part 3: EOS Regression
## Closing the Gap to Lab Bubble Point

**Series:** Black Oil PVT from Scratch
**Dataset:** Volve field (Equinor open data)
**Objective:** Tune Tc of C16+ pseudo-component to match CCE bubble point of 213.1 bar

---

### Context

Parts 1 and 2 built the full PR3 EOS engine from first principles.
The untuned result was **200.84 bar** at 107°C.
The lab (CCE experiment) measured **213.1 bar**.
This notebook closes that 12.3 bar gap through single-variable regression.

**Lab data available:** CCE only — bubble point pressure at reservoir temperature.
No Differential Liberation (DL) was performed for this fluid.
Regression is therefore limited to matching Pb — no Bo, Rs, or GOR matching possible.

---

### Dependencies

This notebook is **standalone** — all fluid properties and EOS functions are defined here.
No variables are imported from Part 1 or Part 2.

In [2]:
import numpy as np
import pandas as pd
from scipy.optimize import brentq, minimize_scalar

T = 380.15    # K (107 deg C reservoir temperature)
R = 83.14472  # cm3.bar / (mol.K)

OMEGA_A = 0.45724
OMEGA_B = 0.077796

COMPONENTS = ['N2','CO2','C1','C2','C3','iC4','nC4','iC5','nC5','C6','C7-C15','C16+']
NC = len(COMPONENTS)

# Wellstream composition — Volve 15/9-F-4, corrected wellstream
ZI = np.array([0.0041015, 0.037994, 0.39916,  0.060722, 0.054490, 0.0077129,
               0.028160,  0.010504, 0.016966,  0.023419, 0.19774,  0.15903])

print(f'Composition sum = {ZI.sum():.6f}  (should be 1.0)')

# Critical temperatures in Celsius, converted to Kelvin
# Pure components N2-C6: standard library values
# Pseudocomponents C7-C15 and C16+: Kesler-Lee correlations
TC_C = np.array([-146.95, 31.55, -82.55,  32.28,  96.65, 134.95,
                  152.05, 187.25, 196.45, 234.35, 367.39, 720.07])
TC = TC_C + 273.15

PC = np.array([33.944, 73.866, 46.042, 48.839, 42.455, 36.477,
               37.966, 33.893, 33.701, 30.104, 22.835,  6.9391])  # bar

OMEGA = np.array([0.04,   0.225,  0.013,  0.0986, 0.1524, 0.1848,
                  0.201,  0.227,  0.251,  0.299,  0.48669, 1.3738])

# Peneloux volume shifts — applied to reported volumes only, not fugacity
SSHIFT = np.array([-0.13133424, -0.042730337, -0.14426562, -0.10326835,
                   -0.077501381, -0.061983725, -0.054224897, -0.041772457,
                   -0.030277896, -0.007288776,  0.04733268,  0.49061257])

# Binary interaction parameters — Katz-Firoozabadi defaults
K_IJ = np.zeros((NC, NC))
idx = {c: i for i, c in enumerate(COMPONENTS)}
N2, CO2, C1, C2, C3 = idx['N2'], idx['CO2'], idx['C1'], idx['C2'], idx['C3']
IC4, NC4, IC5, NC5, C6 = idx['iC4'], idx['nC4'], idx['iC5'], idx['nC5'], idx['C6']
C715, C16 = idx['C7-C15'], idx['C16+']

K_IJ[N2, CO2] = K_IJ[CO2, N2] = -0.012
for j in [C1,C2,C3,IC4,NC4,IC5,NC5,C6,C715,C16]:
    K_IJ[N2,j]  = K_IJ[j,N2]  = 0.10
    K_IJ[CO2,j] = K_IJ[j,CO2] = 0.10
K_IJ[C1,C6]   = K_IJ[C6,C1]   = 0.0279
K_IJ[C1,C715] = K_IJ[C715,C1] = 0.04176
K_IJ[C1,C16]  = K_IJ[C16,C1]  = 0.06752
for i in [C2,C3]:
    for j in [C6,C715,C16]:
        K_IJ[i,j] = K_IJ[j,i] = 0.01

print(f'Loaded {NC} components at T = {T} K')

Composition sum = 0.999999  (should be 1.0)
Loaded 12 components at T = 380.15 K


## PR3 EOS Functions

Functions accept `TC_in` as a parameter so the regression loop can vary critical temperatures without redefining anything.

In [3]:
def compute_ai_bi(TC_in):
    """PR3 pure-component a_i (cm6.bar/mol2) and b_i (cm3/mol)."""
    TR_loc = T / TC_in
    KAPPA_loc = np.where(
        OMEGA <= 0.491,
        0.37464 + 1.54226*OMEGA - 0.26992*OMEGA**2,
        0.379642 + 1.48503*OMEGA - 0.164423*OMEGA**2 + 0.016666*OMEGA**3
    )
    ALPHA_loc = (1.0 + KAPPA_loc * (1.0 - np.sqrt(TR_loc)))**2
    a_i = OMEGA_A * R**2 * TC_in**2 / PC * ALPHA_loc
    b_i = OMEGA_B * R * TC_in / PC
    return a_i, b_i


def solve_Z(A_loc, B_loc, phase):
    """
    Solve the PR cubic for Z-factor.

    Cubic is solved with B_raw (unshifted). The Peneloux volume shift is
    applied post-solve to reported Z and V only — not to the cubic itself.
    """
    coeffs = [1.0,
              (B_loc - 1.0),
              (A_loc - 2.0*B_loc - 3.0*B_loc**2),
              (B_loc**3 + B_loc**2 - A_loc*B_loc)]
    roots = np.roots(coeffs)
    real_r = roots[np.abs(roots.imag) < 1e-8].real
    valid_r = np.sort(real_r[real_r > B_loc])
    if len(valid_r) == 0:
        return B_loc + 1e-8
    elif len(valid_r) == 1:
        return valid_r[0]
    else:
        return valid_r[0] if phase == 'L' else valid_r[-1]


def ln_phi_calc(z, P_in, phase, TC_in=None):
    """
    Log fugacity coefficients for composition z at pressure P_in.

    Volume shift cancels in phi_L/phi_V and does not affect K-values or
    bubble point pressure (Peneloux, 1982).
    """
    if TC_in is None:
        TC_in = TC

    a_i, b_i = compute_ai_bi(TC_in)
    a_ij_loc = np.outer(np.sqrt(a_i), np.sqrt(a_i)) * (1.0 - K_IJ)
    a_mix = float(np.einsum('i,ij,j', z, a_ij_loc, z))
    b_mix = float(np.dot(z, b_i))

    A_loc = a_mix * P_in / (R * T)**2
    B_loc = b_mix * P_in / (R * T)
    Z = solve_Z(A_loc, B_loc, phase)

    SQ2 = np.sqrt(2.0)
    da_i = 2.0 * (a_ij_loc @ z)

    lnphi = (
          (b_i / b_mix) * (Z - 1.0)
        - np.log(Z - B_loc)
        - (A_loc / (2.0 * SQ2 * B_loc))
        * (da_i / a_mix - b_i / b_mix)
        * np.log((Z + (1.0+SQ2)*B_loc) / (Z + (1.0-SQ2)*B_loc))
    )
    return lnphi, Z


def bubble_point_residual(P_in, z, TC_in=None):
    """
    Returns sum(K_i * z_i) - 1, which is zero at bubble point.

    Liquid composition equals the feed at bubble point. Wilson K-values
    seed the successive substitution loop; converges to 1e-12 tolerance.
    """
    if TC_in is None:
        TC_in = TC

    K = (PC / P_in) * np.exp(5.373 * (1.0 + OMEGA) * (1.0 - TC_in / T))

    for _ in range(200):
        y = K * z
        y /= y.sum()
        lnphi_L, _ = ln_phi_calc(z, P_in, 'L', TC_in)
        lnphi_V, _ = ln_phi_calc(y, P_in, 'V', TC_in)
        K_new = np.exp(lnphi_L - lnphi_V)
        err = np.sum((np.log(K_new / K))**2)
        K = K_new
        if err < 1e-12:
            break

    return float(np.sum(K * z) - 1.0)

print('EOS functions ready.')

EOS functions ready.


## Verify Untuned Bubble Point

Confirms the Part 2 result before regression. Expected: 200.84 bar.

In [4]:
print('Untuned bubble point check\n')

P_scan = np.arange(50, 260, 10)
res = [bubble_point_residual(float(p), ZI) for p in P_scan]

for p, r in zip(P_scan, res):
    marker = ' <- sign change' if p > 50 and res[list(P_scan).index(p)-1]*r < 0 else ''
    print(f'  P = {p:5.0f} bar   residual = {r:+.5f}{marker}')

P_lo, P_hi = None, None
for k in range(len(res)-1):
    if res[k] * res[k+1] < 0:
        P_lo = float(P_scan[k])
        P_hi = float(P_scan[k+1])
        break

if P_lo is not None:
    P_bub_untuned = brentq(bubble_point_residual, P_lo, P_hi, args=(ZI,), xtol=0.01)
    print(f'\n  Untuned Pbub = {P_bub_untuned:.2f} bar')
    print(f'  Expected     = 200.84 bar')
    print(f'  Match: {"yes" if abs(P_bub_untuned - 200.84) < 0.1 else "no — check inputs"}')
else:
    print('  No sign change found — check inputs')

Untuned bubble point check

  P =    50 bar   residual = +1.45594
  P =    60 bar   residual = +1.12302
  P =    70 bar   residual = +0.88637
  P =    80 bar   residual = +0.70990
  P =    90 bar   residual = +0.57355
  P =   100 bar   residual = +0.46530
  P =   110 bar   residual = +0.37748
  P =   120 bar   residual = +0.30499
  P =   130 bar   residual = +0.24430
  P =   140 bar   residual = +0.19289
  P =   150 bar   residual = +0.14891
  P =   160 bar   residual = +0.11097
  P =   170 bar   residual = +0.07802
  P =   180 bar   residual = +0.04924
  P =   190 bar   residual = +0.02398
  P =   200 bar   residual = +0.00174
  P =   210 bar   residual = -0.01791 <- sign change
  P =   220 bar   residual = -0.03528
  P =   230 bar   residual = -0.05065
  P =   240 bar   residual = -0.06422
  P =   250 bar   residual = -0.07616

  Untuned Pbub = 200.84 bar
  Expected     = 200.84 bar
  Match: yes


## Regression: Tc(C16+) to Match Bubble Point = 213.1 bar

**Variable selected:** Tc of C16+ only.

C16+ accounts for 15.9 mol% and dominates A_MIX through a large a_i. Raising Tc increases alpha, which raises a_i, A_MIX, and bubble point pressure — the highest-leverage single parameter available.

**Monotonicity constraint:** Tc(C7-C15) = 640.54 K < Tc(C16+) < 1400 K

**Solver structure:** `minimize_scalar` (Brent bounded) over Tc(C16+), with `brentq` over pressure inside and successive substitution for K-values at the innermost level.

In [5]:
TARGET_PBUB = 213.1    # bar — CCE lab measurement
TC_C16_BASE = 993.22   # K   — untuned Kesler-Lee value
TC_MONO_MIN = 640.54   # K   — lower bound, equal to Tc(C7-C15)
TC_MONO_MAX = 1400.0   # K


def pbub_from_Tc_C16(Tc_C16_new):
    """Bubble point pressure with only Tc(C16+) modified."""
    TC_reg = TC.copy()
    TC_reg[11] = Tc_C16_new
    return brentq(bubble_point_residual, 50, 350, args=(ZI, TC_reg), xtol=0.01)


def objective(Tc_C16_new):
    try:
        return (pbub_from_Tc_C16(Tc_C16_new) - TARGET_PBUB)**2
    except:
        return 1e10


# Sensitivity scan
print('Sensitivity: Tc(C16+) vs Pbub\n')
print(f'  {"Tc_C16+ (K)":>14s}  {"Pbub (bar)":>12s}  {"Delta (bar)":>12s}')

for Tc_test in [993.22, 1020, 1050, 1080, 1100, 1120, 1150, 1200]:
    try:
        pb = pbub_from_Tc_C16(float(Tc_test))
        print(f'  {Tc_test:>14.2f}  {pb:>12.2f}  {pb - TARGET_PBUB:>+12.2f}')
    except Exception as e:
        print(f'  {Tc_test:>14.2f}  {"failed":>12s}  {str(e)}')

print('\nSolving for exact match...')

result = minimize_scalar(
    objective,
    bounds=(TC_MONO_MIN + 1.0, TC_MONO_MAX),
    method='bounded',
    options={'xatol': 0.01}
)

Tc_C16_tuned = result.x
Pbub_tuned = pbub_from_Tc_C16(Tc_C16_tuned)
change_pct = (Tc_C16_tuned / TC_C16_BASE - 1.0) * 100

print(f'\n  Tc(C16+) untuned = {TC_C16_BASE:.2f} K')
print(f'  Tc(C16+) tuned   = {Tc_C16_tuned:.2f} K  ({change_pct:+.2f}%)')
print(f'  Change           = {Tc_C16_tuned - TC_C16_BASE:+.2f} K')
print()
print(f'  Pbub untuned = {P_bub_untuned:.2f} bar')
print(f'  Pbub tuned   = {Pbub_tuned:.2f} bar')
print(f'  Target (CCE) = {TARGET_PBUB:.1f} bar')
print(f'  Delta        = {Pbub_tuned - TARGET_PBUB:+.3f} bar')
print()

if change_pct < 15.0:
    print(f'  Change within 15% — physically reasonable')
    print(f'  Monotonicity OK: Tc(C7-C15) = {TC_MONO_MIN} K < Tc(C16+) = {Tc_C16_tuned:.1f} K')
else:
    print('  Warning: change > 15% — consider Tc(C7-C15) as a second regression variable')

Sensitivity: Tc(C16+) vs Pbub

     Tc_C16+ (K)    Pbub (bar)   Delta (bar)
          993.22        200.84        -12.26
         1020.00        206.41         -6.69
         1050.00        212.68         -0.42
         1080.00        219.02         +5.92
         1100.00        223.27        +10.17
         1120.00        227.54        +14.44
         1150.00        234.01        +20.91
         1200.00        244.95        +31.85

Solving for exact match...

  Tc(C16+) untuned = 993.22 K
  Tc(C16+) tuned   = 1051.98 K  (+5.92%)
  Change           = +58.76 K

  Pbub untuned = 200.84 bar
  Pbub tuned   = 213.10 bar
  Target (CCE) = 213.1 bar
  Delta        = +0.000 bar

  Change within 15% — physically reasonable
  Monotonicity OK: Tc(C7-C15) = 640.54 K < Tc(C16+) = 1052.0 K


## K-values and Phase Compositions at Tuned Bubble Point

In [6]:
TC_tuned = TC.copy()
TC_tuned[11] = Tc_C16_tuned

lnphi_L, Z_L_bp = ln_phi_calc(ZI, Pbub_tuned, 'L', TC_tuned)
K_final = (PC / Pbub_tuned) * np.exp(5.373 * (1 + OMEGA) * (1 - TC_tuned / T))

for _ in range(200):
    y = K_final * ZI
    y /= y.sum()
    lnphi_V, Z_V_bp = ln_phi_calc(y, Pbub_tuned, 'V', TC_tuned)
    K_new = np.exp(lnphi_L - lnphi_V)
    if np.sum((np.log(K_new / K_final))**2) < 1e-12:
        break
    K_final = K_new

y_bp = K_final * ZI
y_bp /= y_bp.sum()

# Peneloux-shifted Z for comparison with commercial software
a_i_tuned, b_i_tuned = compute_ai_bi(TC_tuned)
C_I_tuned = SSHIFT * b_i_tuned
C_MIX_tuned = float(np.dot(ZI, C_I_tuned))
Z_L_shift = Z_L_bp - C_MIX_tuned * Pbub_tuned / (R * T)

print(f'Bubble point = {Pbub_tuned:.2f} bar  (target: {TARGET_PBUB} bar)')
print(f'Z_L raw      = {Z_L_bp:.5f}')
print(f'Z_L shifted  = {Z_L_shift:.5f}')
print(f'Z_V raw      = {Z_V_bp:.5f}')
print()

df_kv = pd.DataFrame({
    'z_i (feed)':   np.round(ZI,      5),
    'y_i (vapour)': np.round(y_bp,    5),
    'K_i':          np.round(K_final, 5),
}, index=COMPONENTS)

print('K-values and incipient vapour composition:')
display(df_kv)

print('\nComparison with commercial software reference:')
ref_K = {
    'N2': 2.8917, 'CO2': 1.3322, 'C1': 1.9975, 'C2': 1.0364,
    'C3': 0.7029, 'C6': 0.2116, 'C7-C15': 0.0419, 'C16+': 4.49e-6
}
for comp, kref in ref_K.items():
    i = COMPONENTS.index(comp)
    dev = (K_final[i] - kref) / kref * 100 if kref > 1e-8 else 0.0
    print(f'  {comp:8s}  K_calc={K_final[i]:.5f}  K_ref={kref:.5f}  dev={dev:+.2f}%')

Bubble point = 213.10 bar  (target: 213.1 bar)
Z_L raw      = 1.67607
Z_L shifted  = 1.16455
Z_V raw      = 0.86951

K-values and incipient vapour composition:


,z_i (feed),y_i (vapour),K_i
N2,0.00410,0.01172,2.85869
CO2,0.03799,0.05034,1.32498
C1,0.39916,0.79412,1.98947
C2,0.06072,0.06334,1.04317
C3,0.05449,0.03911,0.71781
iC4,0.00771,0.00411,0.53249
nC4,0.02816,0.01322,0.46931
iC5,0.01050,0.00379,0.36054
nC5,0.01697,0.00563,0.33160
C6,0.02342,0.00527,0.22513



Comparison with commercial software reference:
  N2        K_calc=2.85869  K_ref=2.89170  dev=-1.14%
  CO2       K_calc=1.32498  K_ref=1.33220  dev=-0.54%
  C1        K_calc=1.98947  K_ref=1.99750  dev=-0.40%
  C2        K_calc=1.04317  K_ref=1.03640  dev=+0.65%
  C3        K_calc=0.71781  K_ref=0.70290  dev=+2.12%
  C6        K_calc=0.22513  K_ref=0.21160  dev=+6.40%
  C7-C15    K_calc=0.04730  K_ref=0.04190  dev=+12.88%
  C16+      K_calc=0.00000  K_ref=0.00000  dev=-61.63%


## Summary

In [7]:
MW_arr = np.array([28.013, 44.01, 16.043, 30.07, 44.097, 58.124,
                   58.124, 72.151, 72.151, 84.0, 150.0, 480.0])
MW_MIX = float(np.dot(ZI, MW_arr))

print('REGRESSION SUMMARY')
print()
print('  Parameter adjusted:')
print(f'    Tc(C16+)  {TC_C16_BASE:.2f} K  ->  {Tc_C16_tuned:.2f} K  ({change_pct:+.2f}%)')
print()
print('  Unchanged: TC (C1-C15), PC, OMEGA, BIPs, volume shifts')
print()
print('  Results:')
print(f'    Pbub untuned = {P_bub_untuned:.2f} bar')
print(f'    Pbub tuned   = {Pbub_tuned:.2f} bar')
print(f'    Target (CCE) = {TARGET_PBUB:.1f} bar')
print(f'    Delta        = {Pbub_tuned - TARGET_PBUB:+.3f} bar')
print()
print('  Physical checks:')
print(f'    Monotonicity: Tc(C7-C15) = {TC_MONO_MIN} K < Tc(C16+) = {Tc_C16_tuned:.1f} K  OK')
print(f'    Change < 15%: {change_pct:.2f}%  OK')
print(f'    MW_mix:       {MW_MIX:.3f} g/mol')
print()
print('  Note: model validated against one observation (CCE Pb only).')
print('  DL data needed to extend matching to Bo, Rs, and GOR.')

REGRESSION SUMMARY

  Parameter adjusted:
    Tc(C16+)  993.22 K  ->  1051.98 K  (+5.92%)

  Unchanged: TC (C1-C15), PC, OMEGA, BIPs, volume shifts

  Results:
    Pbub untuned = 200.84 bar
    Pbub tuned   = 213.10 bar
    Target (CCE) = 213.1 bar
    Delta        = +0.000 bar

  Physical checks:
    Monotonicity: Tc(C7-C15) = 640.54 K < Tc(C16+) = 1052.0 K  OK
    Change < 15%: 5.92%  OK
    MW_mix:       124.449 g/mol

  Note: model validated against one observation (CCE Pb only).
  DL data needed to extend matching to Bo, Rs, and GOR.
